# FlyArystan turnaround bottleneck analysis

This notebook uses a reproducible simulated operational log anchored to public evidence. It tests whether tight scheduled turnarounds are associated with larger turnaround delays and whether delayed incoming turns are associated with severe downstream departure delays.

**Sources:** [Bizmedia OTP article](https://bizmedia.kz/2026-09-10-kazahstanu-est-chem-gorditsya-flyarystan-prevzoshel-srednij-pokazatel-punktualnosti-evropejskih-aviakompanij/), [Kapital.kz fine report](https://kapital.kz/gosudarstvo/128470/flyarystan-vyplatit-shtraf-v-razmere-3-6-mln-tenge.html), [Aviation Administration of Kazakhstan](https://caa.gov.kz/ru/blog/post/vedetsya-sistemnaya-i-planomernaya-rabota-po-razvitiyu-otrasli-grazhdanskoj-aviacii), [Consumer Protection Committee](https://www.gov.kz/org/mti-kzpp/press/news/details/1149776?lang=ru).

## 1. Imports and cleaning

`data_raw.csv` contains 1,204 rows: 1,200 simulated flight records plus four exact duplicate log records. It also contains mixed date, unit, spacing, and currency formatting. The cleaning sequence below is deterministic and uses the unique flight ID to remove duplicate logs.

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind, chi2_contingency

raw = pd.read_csv('data_raw.csv')
raw.shape


(1204, 12)

In [2]:
df = raw.copy()
df['flight_id'] = df['flight_id'].astype('string').str.strip()
df = df.drop_duplicates(subset='flight_id', keep='first').copy()
df['flight_date'] = pd.to_datetime(df['flight_date'], format='mixed', dayfirst=True, errors='coerce')
for col in ['scheduled_turnaround_mins','actual_turnaround_mins','turnaround_delay_mins','departure_delay_mins','downstream_severe_delay_flag']:
    df[col] = pd.to_numeric(df[col].astype('string').str.extract(r'(-?\d+(?:\.\d+)?)')[0], errors='coerce')
df['compensation_cost_kzt'] = pd.to_numeric(df['compensation_cost_kzt'].astype('string').str.replace(',','',regex=False).str.extract(r'(\d+(?:\.\d+)?)')[0], errors='coerce')
df['route'] = df['route'].astype('string').str.strip()
df['aircraft_tail_number'] = df['aircraft_tail_number'].astype('string').str.strip()
df['turnaround_window_type'] = np.where(df['scheduled_turnaround_mins'] <= 35, 'Tight', 'Standard')
df['turnaround_delay_mins'] = (df['actual_turnaround_mins'] - df['scheduled_turnaround_mins']).clip(lower=0).round(1)
df['incoming_delay_flag'] = df['incoming_delay_flag'].astype('string').str.strip()
df['downstream_severe_delay_flag'] = (df['departure_delay_mins'] > 30).astype(int)
df = df.sort_values('flight_id').reset_index(drop=True)
assert df.shape == (1200,12)
assert df['flight_id'].is_unique and df.isna().sum().sum() == 0
df.head()


,flight_id,flight_date,route,aircraft_tail_number,scheduled_turnaround_mins,actual_turnaround_mins,turnaround_delay_mins,departure_delay_mins,turnaround_window_type,incoming_delay_flag,downstream_severe_delay_flag,compensation_cost_kzt
0,FA-00001,2026-06-20,NQZ-SCO,UP-A3205,31,58.4,27.4,23.4,Tight,On-Time,0,47161
1,FA-00002,2026-08-28,ALA-NQZ,UP-A3208,47,54.0,7.0,5.0,Standard,On-Time,0,13892
2,FA-00003,2026-06-14,ALA-NQZ,UP-A3202,56,70.3,14.3,9.1,Standard,On-Time,0,24152
3,FA-00004,2026-08-28,ALA-UKK,UP-A3208,36,51.4,15.4,14.5,Standard,On-Time,0,32326
4,FA-00005,2026-07-20,ALA-AKT,UP-A3202,56,70.7,14.7,0.5,Standard,On-Time,0,5695


## 2. Descriptive statistics

The target measures are turnaround delay minutes and the downstream severe-delay flag. The operational split is fixed before testing: Tight means scheduled turnaround ≤35 minutes; Standard means >35 minutes. Incoming delay is Delayed when the incoming turnaround exceeded 15 minutes.

In [3]:
df.groupby('turnaround_window_type')['turnaround_delay_mins'].agg(['count','mean','median','std','max'])


,count,mean,median,std,max
turnaround_window_type,,,,,
Standard,943,16.320891,15.6,7.093566,50.3
Tight,257,28.567315,27.3,7.450395,52.7


In [4]:
pd.crosstab(df['incoming_delay_flag'], df['downstream_severe_delay_flag'], margins=True)


downstream_severe_delay_flag,0,1,All
incoming_delay_flag,,,
Delayed,115,104,219
On-Time,979,2,981
All,1094,106,1200


## 3. Hypothesis pair 1: turnaround window

**H0_1:** Mean turnaround delay is identical for Tight (≤35 minutes) and Standard (>35 minutes) scheduled turnarounds.

**H1_1:** Mean turnaround delay is higher for Tight turnarounds. The independent Welch two-sample t-test uses α=0.05 and a one-sided alternative because the direction is specified in advance.

In [5]:
tight = df.loc[df['turnaround_window_type']=='Tight','turnaround_delay_mins']
standard = df.loc[df['turnaround_window_type']=='Standard','turnaround_delay_mins']
t_test = ttest_ind(tight, standard, equal_var=False, alternative='greater')
print({'tight_mean':tight.mean(), 'standard_mean':standard.mean(), 'difference':tight.mean()-standard.mean(), 't_statistic':t_test.statistic, 'p_value':t_test.pvalue, 'verdict':'Reject H0_1' if t_test.pvalue < 0.05 else 'Do not reject H0_1'})


{'tight_mean': np.float64(28.567315175097274), 'standard_mean': np.float64(16.320890774125132), 'difference': np.float64(12.246424400972142), 't_statistic': np.float64(23.59684473761142), 'p_value': np.float64(1.5862850521831535e-77), 'verdict': 'Reject H0_1'}


## 4. Hypothesis pair 2: delay propagation

**H0_2:** Severe downstream departure delay (>30 minutes) is independent of whether the incoming turnaround was delayed (>15 minutes).

**H1_2:** Flights with delayed incoming turnarounds have a higher severe-delay rate. The chi-square test of independence uses α=0.05.

In [6]:
table = pd.crosstab(df['incoming_delay_flag'], df['downstream_severe_delay_flag']).reindex(index=['On-Time','Delayed'], columns=[0,1], fill_value=0)
chi2, p_value, dof, expected = chi2_contingency(table)
rate = df.groupby('incoming_delay_flag')['downstream_severe_delay_flag'].mean()
print(table)
print({'chi2_statistic':chi2, 'p_value':p_value, 'dof':dof, 'on_time_rate':rate['On-Time'], 'delayed_rate':rate['Delayed'], 'verdict':'Reject H0_2' if p_value < 0.05 else 'Do not reject H0_2'})


downstream_severe_delay_flag    0    1
incoming_delay_flag                   
On-Time                       979    2
Delayed                       115  104
{'chi2_statistic': np.float64(491.2100879643142), 'p_value': np.float64(7.771617492867133e-109), 'dof': 1, 'on_time_rate': np.float64(0.0020387359836901123), 'delayed_rate': np.float64(0.4748858447488584), 'verdict': 'Reject H0_2'}


## 5. Limitations and decision

The simulation does not contain weather severity, maintenance defects, crew legality, gate availability, passenger connection load, or airport-level handling timestamps. These omitted factors could confound the relationship between turnaround windows and delay propagation. The results support testing a 45-minute peak turnaround buffer or decoupling the tightest rotations in a controlled operational pilot; they do not estimate causal savings for the real airline without flight-level logs.